# 02 — Load HuggingFace train + dedup (guardrail) vs Kaggle

Carica il dataset HF (train), lo normalizza e (opzionale) lo deduplica contro Kaggle convertito.

Output: `data/processed/hf_train_dedup.jsonl`

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

kaggle_jsonl = PROC_DIR / 'kaggle_hf_like.jsonl'
kaggle_jsonl

WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/kaggle_hf_like.jsonl')

In [2]:
import sys
sys.path.append(str((PROJECT_ROOT / 'src').resolve()))

from data.hf_loader import HFLoadConfig, load_hf_puzzles, save_jsonl
from data.jsonl_io import read_jsonl
from utils.hashing import puzzle_hash


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# TODO: inserisci qui l'ID del tuo dataset HF (train)
HF_DATASET_ID = 'tm21cy/NYT-Connections'
cfg = HFLoadConfig(dataset_id=HF_DATASET_ID, split='train', num_permutations=1, seed=1234)


In [4]:
hf_puzzles = load_hf_puzzles(cfg)
print('HF puzzles:', len(hf_puzzles))
hf_puzzles[0]

HF puzzles: 652


{'puzzle_id': 'hf_0',
 'date': '2024-06-03 00:00:00',
 'words': ['LASER',
  'PLUCK',
  'THREAD',
  'WAX',
  'COIL',
  'SPOOL',
  'WIND',
  'WRAP',
  'HONEYCOMB',
  'ORGANISM',
  'SOLAR PANEL',
  'SPREADSHEET',
  'BALL',
  'MOVIE',
  'SCHOOL',
  'VITAMIN'],
 'answers': [{'answerDescription': 'REMOVE, AS BODY HAIR',
   'words': ['LASER', 'PLUCK', 'THREAD', 'WAX']},
  {'answerDescription': 'TWIST AROUND',
   'words': ['COIL', 'SPOOL', 'WIND', 'WRAP']},
  {'answerDescription': 'THINGS MADE OF CELLS',
   'words': ['HONEYCOMB', 'ORGANISM', 'SOLAR PANEL', 'SPREADSHEET']},
  {'answerDescription': 'B-___',
   'words': ['BALL', 'MOVIE', 'SCHOOL', 'VITAMIN']}],
 'metadata': {'source': 'hf', 'split': 'train'}}

In [5]:
if kaggle_jsonl.exists():
    kaggle = read_jsonl(kaggle_jsonl)
    kaggle_hashes = {puzzle_hash(p['words']) for p in kaggle}
    before = len(hf_puzzles)
    hf_puzzles = [p for p in hf_puzzles if puzzle_hash(p['words']) not in kaggle_hashes]
    print('Dedup removed:', before - len(hf_puzzles))
else:
    print('Kaggle JSONL not found -> skip dedup')


Dedup removed: 0


In [6]:
out_path = PROC_DIR / 'hf_train_dedup.jsonl'
save_jsonl(hf_puzzles, out_path)
print('Saved:', out_path)
print('Bytes:', out_path.stat().st_size)

Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\hf_train_dedup.jsonl
Bytes: 420618
